# Gated Recurrent Unit: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.gru_wrapper import GRUModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "gru"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El protocolo Walk-Forward seleccionó una arquitectura GRU con ventana de 7 días, 128 neuronas y 1 capa.
- La flexibilidad de la red derivó en una alta varianza en las predicciones, oscilando de forma pronunciada en torno a la volatilidad realizada.
- La naturaleza recurrente de sus compuertas le otorgó una alta reactividad a los cambios recientes, pero limitó su capacidad para filtrar el ruido de corto plazo.
- Fuera de la muestra de entrenamiento, el modelo mostró menor capacidad predictiva que los modelos paramétricos (QLIKE = 0.5484) y dejó efectos ARCH remanentes a corto y mediano plazo.
- A pesar de su flexibilidad no paramétrica, la GRU no logró calibrar la magnitud máxima del shock extremo de 2026, evidenciando limitaciones ante cambios abruptos de régimen.
:::

## Introducción

La Unidad Recurrente con Compuertas (*Gated Recurrent Unit*, GRU) es una arquitectura de red neuronal recurrente diseñada para modelar dependencias temporales en datos secuenciales. A diferencia de las redes neuronales tradicionales, incorpora mecanismos de memoria que permiten conservar y actualizar información de observaciones anteriores mediante compuertas entrenables.

En el contexto de las series financieras, esta arquitectura ofrece una alternativa a los modelos econométricos tradicionales, al permitir que la relación entre los retornos históricos y la volatilidad futura sea aprendida directamente a partir de los datos. Su estructura recurrente facilita la identificación de patrones temporales y cambios en la dinámica de los retornos, sin establecer previamente una ecuación paramétrica para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo GRU para la predicción de la volatilidad del crudo Brent bajo el protocolo del benchmark. Se estudia el efecto de distintas longitudes de secuencia sobre el desempeño predictivo de la red, manteniendo las condiciones de información y evaluación establecidas para los demás modelos del proyecto.

### Arquitectura

Una red GRU procesa secuencialmente las observaciones de entrada y actualiza un estado oculto que resume la información temporal disponible. Para controlar este proceso, utiliza dos mecanismos principales: la compuerta de actualización (*update gate*) y la compuerta de reinicio (*reset gate*).

<div align="center">
  <img src="../diagrams/gru_cell_diagram.jpeg" alt="Diagrama de Celda GRU" width="80%">
</div>

La compuerta de actualización determina qué proporción de la información anterior debe conservarse, mientras que la compuerta de reinicio regula la influencia del estado previo en la construcción de nueva información. Para una observación de entrada $\mathbf{x}_t$, estas operaciones se expresan como:

$$
\mathbf{z}_t = \sigma\left(\mathbf{W}_z\mathbf{x}_t+\mathbf{U}_z\mathbf{h}_{t-1}+\mathbf{b}_z\right),
$$

$$
\mathbf{r}_t = \sigma\left(\mathbf{W}_r\mathbf{x}_t+\mathbf{U}_r\mathbf{h}_{t-1}+\mathbf{b}_r\right),
$$

donde $\mathbf{z}_t$ representa la compuerta de actualización, $\mathbf{r}_t$ la compuerta de reinicio, $\sigma$ la función sigmoide y $\mathbf{h}_{t-1}$ el estado oculto anterior.

El estado candidato se calcula mediante:

$$
\tilde{\mathbf{h}}_t =
\tanh\left(
\mathbf{W}_h\mathbf{x}_t+
\mathbf{U}_h(\mathbf{r}_t\odot\mathbf{h}_{t-1})+
\mathbf{b}_h
\right),
$$

y el nuevo estado oculto se obtiene como:

$$
\mathbf{h}_t =
(1-\mathbf{z}_t)\odot\mathbf{h}_{t-1}
+
\mathbf{z}_t\odot\tilde{\mathbf{h}}_t,
$$

donde $\odot$ representa el producto de Hadamard (elemento a elemento). 

Una vez calculado el estado oculto final de la secuencia, $\mathbf{h}_t \in \mathbb{R}^{H}$ (donde $H$ es el número de unidades ocultas), este vector resume la información de la ventana temporal. Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa (*fully connected layer*) con pesos $\mathbf{w} \in \mathbb{R}^{H}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva (un requisito matemático para la volatilidad), el resultado escalar $y_t$ pasa por una función de activación *Softplus*, la cual es una versión suavizada de la función ReLU:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) = \ln(1 + e^{y_t}),
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las unidades GRU como de la capa de salida.

<div align="center">
  <img src="../diagrams/gru_arquitecture.jpeg" alt="Diagrama de Arquitectura GRU" width="80%">
</div>

Estos mecanismos permiten que la red aprenda qué información temporal conservar, modificar o descartar. A diferencia del perceptrón multicapa (*Multilayer Perceptron*, MLP), que recibe los rezagos como un vector de características, la GRU procesa las observaciones como una secuencia ordenada, incorporando explícitamente la dependencia temporal en su arquitectura.

Para aplicar esta arquitectura al pronóstico de la volatilidad del crudo Brent, se partió del retorno logarítmico diario $r_t$. Dado que los retornos financieros presentan magnitudes pequeñas (del orden de $10^{-3}$), su ingreso directo a la red provoca el desvanecimiento de los gradientes durante la retropropagación, impidiendo el aprendizaje. Por esta razón, los retornos fueron previamente estandarizados mediante una transformación Z-score (media cero y varianza unitaria). Para evitar cualquier fuga de información (*data leakage*), el escalador (`StandardScaler`) se ajustó exclusivamente con el primer 80 % de los datos de cada fold de entrenamiento, aplicando únicamente la transformación sobre el 20 % restante de validación y sobre el conjunto de prueba. 

:::{admonition} Información: Transformación Z-score
:class: info

La estandarización Z-score es una técnica de preprocesamiento que transforma una variable para que tenga una media de cero y una desviación estándar de uno. Dado un retorno logarítmico $r_t$, su versión estandarizada $\tilde{r}_t$ se calcula mediante:

$$
\tilde{r}_t = \frac{r_t - \mu_{\text{train}}}{\sigma_{\text{train}}}
$$

donde $\mu_{\text{train}}$ y $\sigma_{\text{train}}$ son la media y la desviación estándar calculadas sobre el conjunto de entrenamiento. 
:::

Sea $\tilde{r}_t$ el retorno estandarizado, para una ventana temporal de longitud $L$, la secuencia de entrada se define como:

$$
\mathbf{X}_t =
\begin{bmatrix}
\tilde{r}_{t-L}\\
\tilde{r}_{t-L+1}\\
\vdots\\
\tilde{r}_{t-1}
\end{bmatrix}
\in \mathbb{R}^{L\times 1}.
$$

El modelo completo (la GRU y su capa de salida) genera el pronóstico de la varianza condicional del día inmediatamente posterior mediante la función $f_{\boldsymbol{\theta}}(\mathbf{X}_t)$.

El problema se aborda mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consiste en encontrar los parámetros $\boldsymbol{\theta}$ que minimicen la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad. El problema de optimización se expresa como:

$$
\boldsymbol{\theta}^{*}
=\arg\min_{\boldsymbol{\theta}}
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_{\text{QLIKE}}
\left(
RS_{i},
\hat{RS}_{i}
\right),
$$

donde $N$ es el número de secuencias de entrenamiento y $RS_i$ es la varianza de Rogers-Satchell utilizada como *target*.

Los parámetros se actualizan mediante retropropagación a través del tiempo (*backpropagation through time*, BPTT) y el optimizador Adaptive Moment Estimation (Adam). La elección del número de unidades recurrentes, las capas ocultas, la longitud de las secuencias ($L$) y los parámetros de entrenamiento (tasa de aprendizaje, *dropout*) influye directamente sobre la capacidad de ajuste y generalización del modelo.

La implementación de la GRU se realiza utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática para el cálculo de gradientes y la optimización de los pesos.
## Entrenamiento

Para la selección de hiperparámetros del modelo GRU, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional que en los modelos paramétricos. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red neuronal y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento.

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el número de neuronas en la capa oculta (`hidden_size`), la cantidad de capas recurrentes, la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def gru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=GRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=gru_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

Respecto a la eficiencia del algoritmo de búsqueda, este se detuvo en el trial 23 sin alcanzar el límite máximo de 50 evaluaciones, debido a la activación del mecanismo de parada temprana (*Early Stopping*) tras 15 intentos consecutivos sin mejoras. El mecanismo de poda (*Pruning*) descartó 6 configuraciones, lo que evidenció la utilidad del *Median Pruner* para abortar tempranamente arquitecturas con bajo desempeño predictivo durante las primeras particiones, optimizando así el presupuesto computacional. La mejor configuración registró un QLIKE de 0.4565.

El ajuste de los pesos bajo el esquema Walk-Forward tuvo una duración total de 4.83 minutos (289.80 segundos), con un tiempo medio de 3.08 segundos por pliegue. Es fundamental precisar que este costo corresponde exclusivamente al entrenamiento de la red. La evaluación completa del modelo, que incluyó el reentrenamiento iterativo durante la generación de pronósticos (*forecast*) para las observaciones fuera de muestra, elevó el tiempo de entrenamiento total a aproximadamente 7 horas y media. Este alto costo computacional frente a los 9.12 y 12.54 segundos de los modelos GARCH y EGARCH reflejó el compromiso necesario para dotar al modelo de mayor flexibilidad no paramétrica.

In [2]:
resumen_data = []

exec_id = 2
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="GRUModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
23,16,6,0.4565,289.8000,3.0830


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una pérdida QLIKE en el rango de 0.457 a 0.475. Las diferencias entre estas configuraciones fueron menores que la variabilidad entre folds ($\sigma \approx 0.14$), lo que indicó un empate estadístico en su desempeño predictivo. A pesar de que los hiperparámetros continuos (tasa de aprendizaje, *weight decay* y *dropout*) variaron sin un patrón claro, se observó un consenso en la arquitectura base: todas seleccionaron una ventana de memoria (*lookback*) de 7 días, 128 neuronas ocultas y, en su mayoría, una sola capa recurrente.

Esta convergencia arquitectónica aportó información relevante sobre la dinámica de los datos. Aunque el espacio de búsqueda exploró ventanas temporales más largas (14, 21 y 28 días), la red convergió de forma consistente hacia los 7 días. Esto sugirió que la GRU priorizó una memoria de corto plazo (aproximadamente una semana bursátil) para capturar el agrupamiento de volatilidad inmediato, evitando incorporar ruido de periodos más antiguos. De igual forma, la elección del límite superior de 128 neuronas junto con una sola capa evidenció que la red logró un equilibrio al aumentar su capacidad de representación en el espacio latente, pero manteniendo una sola capa recurrente para mitigar el riesgo de sobreajuste.

In [3]:
exec_dir = RESULTS_PATH / "exec_2"

df_gru = pd.read_csv(exec_dir / "GRUModel_results.csv")

df_gru["lr"] = df_gru["learning_rate"].apply(lambda x: f"{x:.4f}")
df_gru["wd"] = df_gru["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "hidden_size", "num_layers", "dropout", "lr", "wd"]
df_agg = df_gru.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "hidden_size": "Neuronas",
    "num_layers": "Capas",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Neuronas", "Capas", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Neuronas,Capas,Dropout,LR,Weight Decay,QLIKE
7,128,1,0.299,0.0065,6.5e-04,0.457 ± 0.145
7,128,1,0.141,0.0074,7.3e-04,0.457 ± 0.141
7,128,1,0.133,0.0040,7.5e-04,0.460 ± 0.152
7,128,2,0.206,0.0025,9.5e-04,0.474 ± 0.157
7,128,1,0.131,0.0003,8.7e-05,0.475 ± 0.123


Las dos configuraciones con menor pérdida compartieron exactamente la misma arquitectura base (Lookback de 7 días, 128 neuronas y 1 capa), difiriendo únicamente en sus hiperparámetros de regularización y tasa de aprendizaje. Sus medias de QLIKE fueron idénticas (0.457). Dado que esta diferencia nula está muy por debajo de la variabilidad observada entre los folds ($\sigma \approx 0.14$), ambas fueron estadísticamente indistinguibles en desempeño predictivo.

Un hallazgo relevante fue la magnitud de la desviación estándar del QLIKE en el GRU, la cual superó a la reportada por los modelos GARCH y EGARCH ($\sigma \approx 0.06$ y $0.10$, respectivamente). Una hipótesis plausible para explicar este fenómeno radica en la naturaleza de los algoritmos de optimización. Mientras que el GARCH utiliza una estimación por Máxima Verosimilitud que converge a un único óptimo, el GRU se entrena mediante descenso de gradiente estocástico sobre una superficie de pérdida no convexa. Esto, sumado a la inicialización aleatoria de los pesos en cada partición (folds), indujo una mayor varianza en los resultados. Adicionalmente, la alta flexibilidad de la red neuronal provocó que su desempeño fuera más volátil ante los distintos regímenes de volatilidad presentes en cada fold de validación.

Ante la equivalencia predictiva en la media, se seleccionó la primera configuración al registrar el mínimo absoluto de QLIKE. Aunque la segunda opción presentó una desviación estándar marginalmente menor, la primera empleó un *dropout* superior (0.299 frente a 0.141) y un *weight decay* moderado ($6.5 \times 10^{-4}$). Esta mayor regularización reduce la probabilidad de sobreajuste en la capa de 128 neuronas, garantizando una mejor capacidad de generalización.

In [17]:
best_model = GRUModel(
    look_back=7, 
    hidden_size=128,
    num_layers=1, 
    dropout=0.299, 
    learning_rate=0.0065,
    weight_decay=6.5e-04, 
    max_epochs=200, 
    patience=20, 
    retrain_freq=1, 
    seed=42
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El análisis de la curva de aprendizaje descarta la presencia de sobreajuste. En un escenario de sobreajuste clásico, la pérdida de entrenamiento continuaría disminuyendo mientras la de validación empezaría a incrementarse, abriendo una brecha progresiva. En este caso, la pérdida de validación no solo se mantiene estable, sino que es consistentemente menor que la de entrenamiento desde el inicio. Esta brecha negativa se explica por el sesgo de selección temporal del conjunto de datos: la partición cronológica destina el periodo de mayor dificultad predictiva (los shocks extremos de 2020) al entrenamiento, mientras que la validación (2021-2022) corresponde a un régimen de volatilidad más moderado y, por tanto, con un QLIKE intrínsecamente más bajo.

In [21]:
tt.plot_learning_curve(
    history=best_model.get_history(), 
    title="Curva de Aprendizaje",
    ylab = "Quasi-Likelihood (QLIKE)", 
    xlab_pad=14, ylab_pad=14,
    line_width=2.0, title_size=14, label_size=14,
    tick_size=12, legend_size=14,
    ylim=(0, 5), xlim=(0, 60), y_step=0.5, x_step=4,
    legend_bbox=(0.95, 0.95)
)

El modelo alcanzó su punto de mínimo error en la época 37, punto en el cual se seleccionaron los mejores pesos. La curva de aprendizaje evidenció que, tras una fase inicial de inestabilidad, las pérdidas de entrenamiento y validación convergieron a partir de la época 15. Esta convergencia, sumada a la baja pérdida inicial en validación, indica que la arquitectura seleccionada (una sola capa y 128 neuronas) logró estabilizarse y operar cerca de su techo de capacidad para este conjunto de datos sin caer en sobreajuste. La red aprendió la tendencia central de la varianza condicional rápidamente, pero la estabilización de la curva en las épocas posteriores sugiere que no logró refinar la predicción de los picos de volatilidad extremos, lo cual es consistente con el desempeño general del modelo no paramétrico.

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no es posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) utilizando la totalidad de la serie de entrenamiento. Dado que la red neuronal fue entrenada con datos estandarizados, la serie `y_train` se transformó previamente utilizando el `StandardScaler` ajustado durante el `fit`. Posteriormente, mediante la clase `TimeSeriesDataset`, se construyó la matriz de rezagos correspondiente, agrupando los retornos en ventanas históricas de 7 días. Estas ventanas se procesaron en un solo lote a través de la red neuronal en modo de evaluación (`eval()`), desactivando el cálculo de gradientes y las capas de regularización. Las varianzas pronosticadas se alinearon temporalmente con los datos reales, descartando los primeros 7 días de la serie (`y_train[look_back:]`) para garantizar que cada predicción correspondiera exactamente con su valor observado.

In [46]:
best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()

train_ds = TimeSeriesDataset(y_train_scaled, rs_train, look_back)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

with torch.no_grad():
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(best_model.device)
        preds_train_var = best_model.net(x_batch).cpu().numpy().flatten()

y_train_aligned = y_train[look_back:]
rs_train_aligned = rs_train[look_back:]

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (In-Sample) como en el de prueba (Out-of-Sample), a partir de las varianzas predichas y los retornos reales, se calcularon los residuos estandarizados mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El análisis de los residuos sobre el conjunto de entrenamiento reveló un ajuste estructural adecuado. Las pruebas de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) no rechazaron sus hipótesis nulas ($p > .05$). Lo anterior indicó que la red logró absorber la dependencia temporal, el agrupamiento de la volatilidad y la estructura no lineal dentro de la muestra. Por otro lado, las pruebas de Jarque-Bera y Lilliefors rechazaron la hipótesis nula de normalidad ($p < .001$), evidenciando la presencia de colas pesadas en los errores. Este comportamiento es característico de los activos financieros y la función de pérdida QLIKE logró asimilarlo sin comprometer la estructura predictiva del modelo.

In [47]:
std_residuals_train = y_train_aligned / np.sqrt(preds_train_var)
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),6.6126,.761,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),27.0856,.133,.399,Sin Autocorrelación
Ljung-Box (Lag 30),37.0823,.175,.524,Sin Autocorrelación
ARCH-LM (Lag 10),7.1052,.715,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),15.9837,.718,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),28.5661,.540,1.000,Sin Efectos ARCH
Jarque-Bera,1412.9858,< .001,--,Residuos No Normales
Lilliefors,0.0586,< .001,--,Residuos No Normales
BDS (Dim 2),0.0419,.967,--,Sin Estructura No Lineal


Los residuos estandarizados del GRU presentaron una estructura consistente con una distribución de colas pesadas, ajustándose a una distribución $t$ de Student. Esta correspondencia se observó en el gráfico Q-Q, donde los cuantiles empíricos siguieron la línea de referencia diagonal a lo largo de casi toda la distribución. No obstante, en los extremos de la distribución (particularmente en la cola derecha) algunos puntos se desviaron de la recta teórica, lo que indicó que la distribución $t$ no logró modelar completamente la magnitud de los eventos más extremos. A pesar de estas desviaciones puntuales, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .4115$; Cramér-von Mises: $p = .3450$) no rechazaron la hipótesis de pertenencia a la distribución $t$.

In [45]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La LRT rechazó la hipótesis nula de igualdad de ajuste ($p < .001$). Los criterios de información evidenciaron una reducción de $\Delta\text{AIC} = 219.91$ y $\Delta\text{BIC} = 214.23$ a favor de la distribución $t$. Esto indicó que la inclusión del parámetro de grados de libertad ($\nu \approx 4.52$) mejoró la descripción de las colas pesadas de los residuos en la muestra de entrenamiento en comparación con el modelo Gaussiano. 

Es importante precisar que, a diferencia de los modelos GARCH y EGARCH, el GRU no asume una distribución paramétrica específica durante su estimación, dado que su función de pérdida (QLIKE) se optimiza directamente sobre la varianza. Por tanto, este contraste estadístico no validó un supuesto interno del modelo, sino que confirmó que la estructura empírica de los residuos del GRU presentó colas pesadas que la distribución gaussiana no capturó de forma adecuada.

In [50]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,205.73","6,415.46","6,426.82",-,-
t-Student,"-3,094.77","6,195.55","6,212.59",221.91,< .001


A diferencia de los modelos paramétricos GARCH y EGARCH, la estructura del GRU logró seguir con mayor precisión la trayectoria de la volatilidad realizada. La inspección visual evidenció que la red neuronal reprodujo el surgimiento de eventos de alta volatilidad, como la crisis del COVID-19 en 2020 y los picos de 2022. En el detalle de la crisis de 2020, se observó que la predicción presentó una ligera subestimación en la magnitud máxima del shock principal (alcanzando aproximadamente 0.18 frente al máximo real de 0.25). A pesar de esta diferencia en la cima, la flexibilidad de la arquitectura permitió capturar el agrupamiento y la dinámica temporal de la volatilidad a lo largo de todo el periodo histórico.

In [41]:
look_back = best_model.context_length

dates_aligned = df_train.index[look_back:]
rs_volatilidad_aligned = np.sqrt(rs_train[look_back:])
pred_volatilidad = np.sqrt(preds_train_var) 

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,      
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (GRU)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

Este nivel de ajuste resultó esperado al tratarse de la muestra de entrenamiento. Al ser evaluada con los mismos datos utilizados para optimizar sus pesos, la red neuronal desplegó su capacidad para mapear la secuencia histórica. Por lo tanto, para determinar si la arquitectura poseyó verdadera capacidad de generalización ante nuevos regímenes de mercado, se procedió a la evaluación sobre el conjunto de prueba (Out-of-Sample).

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre datos no vistos. Mientras que el conjunto de entrenamiento (In-Sample) permite verificar la convergencia del algoritmo a la dinámica histórica, la evaluación Out-of-Sample (conjunto de prueba) determina la utilidad real del modelo ante nuevos regímenes de mercado. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios pre-registrados en el protocolo experimental. Estos criterios se basaron en la diferencia absoluta entre las métricas de prueba y entrenamiento, denotada como:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Se estableció un umbral de tolerancia absoluto de $|\Delta_{abs}| = 0.05$ para todas las métricas de evaluación. Un valor de $\Delta_{abs}$ que superara este margen evidenciaría falta de generalización. 

No obstante, en el contexto de los modelos de volatilidad, es fundamental resaltar la naturaleza asimétrica de la pérdida QLIKE. Dado que esta métrica penaliza severamente la subestimación de la volatilidad, pero es indulgente con la sobreestimación, la interpretación tradicional del sobreajuste debe matizarse. Un modelo que desarrolla un sesgo defensivo (sobreestimando la varianza de forma constante) puede presentar un QLIKE de entrenamiento superior al de prueba. Esto no indica una mejora en la generalización, sino que refleja que el conjunto de prueba presentó regímenes de volatilidad que se acoplaron mejor a dicha sobreestimación. Por lo tanto, el análisis de la brecha $\Delta_{abs}$ se complementó con la inspección de las trayectorias predichas y los diagnósticos de residuos.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` del wrapper del modelo. Este procedimiento aplicó un esquema de ventana expansiva (*rolling forward*) con reentrenamiento periódico, garantizando la ausencia de fuga de información. Las varianzas condicionales predichas se generaron en la escala original para permitir su comparación directa con la varianza realizada.

In [ ]:
start_time = time.time()
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test) 
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": df_test.index, 
    "rs_true": rs_test, 
    "y_pred": preds_test
})

output_path = RESULTS_PATH / "gru_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 15.41 minutos. Esto equivalió a un tiempo medio de aproximadamente 0.97 segundos por predicción (para las 953 observaciones del conjunto de prueba). Este costo es sustancialmente superior al de los modelos paramétricos, cuyo tiempo de inferencia es de milisegundos, reflejando la carga computacional inherente a la actualización iterativa de los pesos en las arquitecturas de Deep Learning.

El análisis de los residuos fuera de muestra presentó un comportamiento mixto. Las pruebas de Ljung-Box no rechazaron la hipótesis nula de autocorrelación ($p\text{-adj} > .05$), y la prueba BDS tampoco rechazó la ausencia de dependencia no lineal ($p = .352$), indicando que el GRU logró capturar la estructura temporal de largo y corto plazo en datos no vistos. Sin embargo, la prueba ARCH-LM detectó efectos ARCH significativos en todos los rezagos evaluados ($p\text{-adj} \le .002$). Esto evidenció que, si bien la red neuronal absorbió la estructura temporal y no lineal de los errores, no logró capturar por completo el agrupamiento de la volatilidad (*volatility clustering*) residual.

In [51]:
preds_path = RESULTS_PATH / "exec_2" / "gru_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(preds_test_var)
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),8.5908,.571,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),26.0390,.165,.494,Sin Autocorrelación
Ljung-Box (Lag 30),30.5887,.436,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),47.8132,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),60.5592,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),61.4721,< .001,.002,Efectos ARCH Significativos
Jarque-Bera,1439.4642,< .001,--,Residuos No Normales
Lilliefors,0.0580,< .001,--,Residuos No Normales
BDS (Dim 2),-0.9300,.352,--,Sin Estructura No Lineal


Este resultado evidenció que, al igual que el EGARCH, el GRU presentó dificultades para capturar los clústeres de volatilidad a corto y mediano plazo. La presencia de efectos ARCH en los rezagos 10, 20 y 30 indicó que los residuos conservaron dependencia condicional durante los días posteriores a un shock. En el caso de la red neuronal, esto se relacionó con la inercia de la arquitectura recurrente: el modelo suavizó la respuesta y reaccionó con un rezago temporal. Por lo tanto, no logró predecir de forma inmediata la agrupación de la volatilidad diaria tras un evento extremo, requiriendo varios pasos para asimilar el cambio de régimen.

El comportamiento de los residuos fuera de muestra fue similar al observado en el conjunto de entrenamiento. El ajuste a la distribución $t$ de Student arrojó un parámetro de grados de libertad de $\nu = 4.4318$. Al ser ligeramente inferior al obtenido In-Sample ($\nu \approx 4.52$), este resultado indicó que los residuos de prueba presentaron colas ligeramente más pesadas. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .9863$; Cramér-von Mises: $p = .9714$) no rechazaron la hipótesis de pertenencia a la distribución $t$. En el gráfico Q-Q, los cuantiles empíricos presentaron una alineación con la línea diagonal a lo largo de casi toda la distribución, con desviaciones puntuales en los valores extremos de la cola derecha.

In [52]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

El modelo GRU no cumplió con el criterio de tolerancia pre-fijado para la métrica principal. La pérdida QLIKE en prueba (0.5484) superó a la de entrenamiento (0.4052), arrojando una diferencia absoluta de $\Delta = +0.1432$, la cual excedió el umbral establecido de 0.05. Asimismo, al comparar este resultado con el QLIKE de validación obtenido durante la búsqueda de hiperparámetros (0.4565), se evidenció una disminución en la capacidad predictiva del modelo fuera de muestra. Esto indicó que, aunque el modelo logró ajustarse a los datos históricos, no generalizó con la misma eficacia ante nuevos regímenes de volatilidad.

In [53]:
look_back = best_model.context_length
rs_train_aligned = rs_train[look_back:] 

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4052,0.5484,+0.1432
CRPS,0.0127,0.0127,-0.0000
Pinball (τ=0.05),0.0028,0.0028,+0.0000
Pinball (τ=0.95),0.0014,0.0018,+0.0004
MSE,0.0000,0.0000,+0.0000
RMSE,0.0019,0.0036,+0.0017
MAE,0.0004,0.0007,+0.0003
HMSE,1.2741,3.2677,+1.9936
HMAE,0.6613,0.8559,+0.1946


Las métricas probabilísticas (CRPS y Pinball) se mantuvieron estables con variaciones marginales, y los errores puntuales (MSE, RMSE, MAE) presentaron incrementos mínimos, indicando que la red capturó la tendencia central. No obstante, las métricas ajustadas por heterocedasticidad presentaron incrementos notables: HMSE (+1.9936) y HMAE (+0.1946). Este resultado fue consistente con la definición del HMSE, el cual pondera los errores por el inverso de la volatilidad. Esto magnificó la desviación durante los días de mayor turbulencia (como el pico de 2026), evidenciando la dificultad del modelo para anticipar la magnitud de eventos extremos.

Este incremento del HMSE evidenció un comportamiento de alta sensibilidad de la red ante las fluctuaciones del mercado. A diferencia de los modelos paramétricos, la GRU no presentó una respuesta suavizada, sino que su predicción exhibió una alta varianza, oscilando de forma pronunciada en torno a la volatilidad realizada. Aunque capturó la presencia del régimen de alta volatilidad en 2026, su ajuste fue errático: en varios días sobreestimó los picos locales, mientras que en el evento máximo del shock, la predicción se mantuvo por debajo de la magnitud real. El HMSE reflejó esta limitación: la red neuronal reaccionó con alta sensibilidad a los cambios de régimen, pero no logró calibrar la amplitud del shock principal.

A diferencia de los modelos paramétricos, que tienden a suavizar los picos debido a su estructura de actualización, la arquitectura GRU demostró una mayor reactividad en sus estimaciones. Al carecer de restricciones matemáticas rígidas, la red intentó adaptarse a cada fluctuación reciente, generando una trayectoria altamente dinámica. No obstante, esta flexibilidad no garantizó una mejor aproximación al shock extremo de 2026. Si bien la red reaccionó a la turbulencia, su incapacidad para filtrar el ruido de corto plazo frente al verdadero cambio de régimen provocó que fallara al estimar la cúspide del evento. Esto evidenció que la flexibilidad no paramétrica, por sí sola, no resolvió la complejidad de los shocks estructurales.

In [55]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(df_preds_test["rs_true"])
pred_test_vol = np.sqrt(df_preds_test["y_pred"])

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (GRU)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La implementación y evaluación del modelo GRU permitieron contrastar la capacidad de esta arquitectura no paramétrica para el pronóstico de la volatilidad del Brent. A diferencia de los modelos de la familia GARCH y EGARCH, la pérdida QLIKE en el conjunto de prueba fue superior (0.5484 frente al rango de 0.40 a 0.50). Asimismo, los diagnósticos de residuos fuera de muestra evidenciaron efectos ARCH significativos en los rezagos 10, 20 y 30, lo que indicó dificultades para capturar por completo los clústeres de volatilidad.

El protocolo de búsqueda de hiperparámetros (Optuna) seleccionó una arquitectura específica: una ventana de memoria (*look_back*) de 7 días (equivalente a una semana bursátil), 128 neuronas ocultas y una sola capa recurrente. A pesar de que el espacio de búsqueda permitía ventanas más largas y configuraciones más profundas, el mejor desempeño en validación se obtuvo con esta estructura. Esto indicó que la red priorizó la memoria de corto plazo para capturar el agrupamiento inmediato de la volatilidad, y optó por aumentar su capacidad de representación mediante la amplitud (128 neuronas) en lugar de la profundidad (múltiples capas), evitando así el riesgo de sobreajuste.

La arquitectura GRU procesó secuencias temporales y capturó la tendencia general del mercado, pero presentó dificultades de calibración. A diferencia de los modelos paramétricos, la red no mostró un efecto de inercia o sesgo defensivo. Por el contrario, su naturaleza recurrente derivó en una alta sensibilidad a las fluctuaciones recientes, generando predicciones con alta varianza. Esta reactividad provocó que la red sobreestimara picos locales y subestimara la magnitud máxima del shock extremo de 2026. Al carecer de restricciones matemáticas, la red no logró distinguir entre el ruido de corto plazo y el verdadero cambio de régimen estructural.

El incremento en las métricas ajustadas por heterocesdasticidad (HMSE y HMAE) reflejó esta falta de calibración durante los días de mayor turbulencia. A pesar de este desempeño inferior en QLIKE, el modelo GRU mostró flexibilidad para adaptarse a los cambios del mercado aprendiendo patrones directamente de los datos. No obstante, los resultados evidenciaron que la flexibilidad no paramétrica del GRU, por sí sola, no superó la eficiencia predictiva de los modelos paramétricos. La ausencia del sesgo inductivo del GARCH (que impone una actualización recursiva y suavizada de la varianza) hizo que el modelo GRU fuera más susceptible a las asimetrías de la función de pérdida y al ruido del mercado.